# 🚦 Smart Toll Pricing System
## Phase 3 — Dynamic Toll Pricing Engine

| | |
|---|---|
| **Student** | Nitin Rajgor |
| **University** | Jain (Deemed-to-be) University, Bengaluru |
| **Paper** | IRE Journals, Vol. 9, Issue 11, May 2026 |

---
### What we do here:
- Load trained Random Forest model
- Build Dynamic Toll Pricing Engine
- Test with real-world traffic scenarios
- Show how prediction + pricing work together

### Research Gap Addressed:
> Existing studies **predict traffic OR optimize toll** separately.  
> This system does **BOTH in one framework** — Paper Section 2.4

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib, json
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

print('✅ Libraries imported!')

---
## 📂 Step 1 — Load Trained Model

In [ ]:
model   = joblib.load('models/rf_model.pkl')
metrics = json.load(open('models/metrics.json'))

print('='*50)
print('  MODEL LOADED')
print('='*50)
print(f'  Model Type : Random Forest Classifier')
print(f'  Trees      : {model.n_estimators}')
print(f'  Features   : {model.n_features_in_}')
print(f'  Accuracy   : {metrics["accuracy"]*100:.2f}%')
print(f'  F1 Score   : {metrics["f1_score"]*100:.2f}%')
print()
print('✅ Model ready for predictions!')

FEATURE_COLS = [
    'hour','day_of_week','month',
    'rush_intensity','time_of_day','season','day_type',
    'traffic_volume','avg_speed','travel_time','vol_speed_ratio',
    'temp_celsius','rain_1h','snow_1h','clouds_all',
    'weather_encoded','bad_weather'
]

---
## 💰 Step 2 — Dynamic Pricing Engine
**Paper Section 3.2.5** — Dynamic Toll Pricing Module

In [ ]:
# Base toll prices per congestion level
BASE_TOLL = {0: 50, 1: 80, 2: 120}
LABELS    = {0: 'Low', 1: 'Medium', 2: 'High'}

def calculate_toll(congestion, volume, speed):
    """
    Dynamic pricing rules:
    - Base price depends on congestion level
    - Demand multiplier if volume is high
    - Speed penalty if traffic is very slow
    """
    base = BASE_TOLL[congestion]

    # Demand multiplier
    if volume > 5000:   multiplier = 1.5
    elif volume > 3500: multiplier = 1.2
    else:               multiplier = 1.0

    # Speed penalty (very slow = surge)
    if speed < 20: multiplier += 0.3

    return round(base * multiplier)


def predict_and_price(hour, day, volume, speed, travel,
                      temp, rain, snow, clouds, weather):
    """
    Full pipeline: Traffic Input → ML Predict → Dynamic Price
    """
    # Engineer features
    rush    = 2 if (7<=hour<=9 or 16<=hour<=18) else (1 if (10<=hour<=11 or 14<=hour<=15) else 0)
    tod     = 0 if 5<=hour<12 else (1 if 12<=hour<17 else (2 if 17<=hour<21 else 3))
    season  = 2
    day_t   = 1 if day>=5 else 0
    vol_sp  = round(volume/(speed+1), 3)
    bad_w   = 1 if (weather>=2 or rain>0 or snow>0) else 0

    features = pd.DataFrame([[
        hour, day, 6, rush, tod, season, day_t,
        volume, speed, travel, vol_sp,
        temp, rain, snow, clouds, weather, bad_w
    ]], columns=FEATURE_COLS)

    pred  = int(model.predict(features)[0])
    proba = model.predict_proba(features)[0]
    toll  = calculate_toll(pred, volume, speed)

    return LABELS[pred], toll, round(max(proba)*100, 1)


print('✅ Dynamic Pricing Engine Ready!')
print()
print('  Pricing Rules:')
print('  ┌─────────────┬──────────┬─────────────────────┐')
print('  │ Congestion  │ Base     │ With Demand Surge   │')
print('  ├─────────────┼──────────┼─────────────────────┤')
print('  │ 🟢 Low      │ ₹50      │ ₹50 – ₹65           │')
print('  │ 🟡 Medium   │ ₹80      │ ₹80 – ₹104          │')
print('  │ 🔴 High     │ ₹120     │ ₹120 – ₹216         │')
print('  └─────────────┴──────────┴─────────────────────┘')

---
## 🧪 Step 3 — Real World Test Scenarios

In [ ]:
scenarios = [
    # name,                    hour, day, vol,  spd, travel, temp, rain, snow, clouds, weather
    ('Monday Morning Rush',      8,   0,  5500,  22,   60,    15,   0,    0,    30,     0),
    ('Tuesday Office Hours',     10,  1,  3200,  55,   32,    18,   0,    0,    40,     1),
    ('Wednesday Afternoon',      14,  2,  2000,  70,   22,    22,   0,    0,    20,     0),
    ('Friday Evening Peak',      18,  4,  4800,  28,   52,    20,   0,    0,    60,     1),
    ('Rainy Thursday Morning',   9,   3,  3800,  35,   45,    12,   5,    0,    90,     2),
    ('Saturday Afternoon',       15,  5,  1800,  75,   18,    25,   0,    0,    10,     0),
    ('Sunday Night Empty Road',  2,   6,  350,   95,   14,    10,   0,    0,    20,     0),
    ('Snowy Monday Morning',     8,   0,  4200,  20,   65,    -2,   0,    3,    100,    3),
]

print('='*65)
print('  REAL WORLD SCENARIOS — PREDICTION + DYNAMIC PRICING')
print('='*65)
print(f"  {'Scenario':<30} {'Congestion':<12} {'Toll':<10} {'Confidence'}")
print('  ' + '-'*60)

results = []
for sc in scenarios:
    name  = sc[0]
    args  = sc[1:]
    label, toll, conf = predict_and_price(*args)
    emoji = {'Low':'🟢','Medium':'🟡','High':'🔴'}[label]
    print(f"  {name:<30} {emoji} {label:<10} ₹{toll:<8} {conf}%")
    results.append({'Scenario':name,'Congestion':label,'Toll':toll})

print('='*65)

---
## 📊 Step 4 — Visualize Pricing Results

In [ ]:
# Chart 1: Toll Price per Scenario
df_res = pd.DataFrame(results)
color_map = {'Low':'#28a745','Medium':'#ffc107','High':'#dc3545'}
bar_colors = [color_map[c] for c in df_res['Congestion']]

fig, ax = plt.subplots(figsize=(14,5))
bars = ax.bar(range(len(df_res)), df_res['Toll'], color=bar_colors, width=0.6, edgecolor='white')
for i,(bar,row) in enumerate(zip(bars, df_res.itertuples())):
    ax.text(i, row.Toll+3, f'₹{row.Toll}', ha='center', fontweight='bold', fontsize=10)
ax.set_xticks(range(len(df_res)))
ax.set_xticklabels(df_res['Scenario'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Toll Price (₹)', fontsize=11)
ax.set_title('Dynamic Toll Price — Different Traffic Scenarios', fontsize=13, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#28a745', label='Low Congestion'),
    mpatches.Patch(color='#ffc107', label='Medium Congestion'),
    mpatches.Patch(color='#dc3545', label='High Congestion')
])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('static/charts/pricing_scenarios.png', dpi=120)
plt.show()
print('💡 Toll price changes dynamically based on congestion level!')

In [ ]:
# Chart 2: Pricing Rules Visual
fig, ax = plt.subplots(figsize=(10,4))

volumes  = list(range(0, 7001, 100))
low_toll = [50 if v<=3500 else 60 for v in volumes]
med_toll = [80 if v<=3500 else 96 for v in volumes]
high_toll= [120 if v<=3500 else (144 if v<=5000 else 180) for v in volumes]

ax.fill_between(volumes, 0, low_toll,  alpha=0.3, color='#28a745', label='Low → ₹50-60')
ax.fill_between(volumes, 0, med_toll,  alpha=0.3, color='#ffc107', label='Medium → ₹80-96')
ax.fill_between(volumes, 0, high_toll, alpha=0.3, color='#dc3545', label='High → ₹120-180')

ax.axvline(x=2500, color='green',  linestyle='--', alpha=0.7, label='Low→Medium (2500)')
ax.axvline(x=4500, color='orange', linestyle='--', alpha=0.7, label='Medium→High (4500)')

ax.set_xlabel('Traffic Volume (veh/hr)', fontsize=11)
ax.set_ylabel('Toll Price (₹)', fontsize=11)
ax.set_title('Dynamic Pricing: Toll Price vs Traffic Volume', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('static/charts/pricing_rules.png', dpi=120)
plt.show()
print('💡 As traffic increases, toll price increases automatically!')

---
## ✅ Summary

In [ ]:
print('='*55)
print('  PRICING ENGINE — COMPLETE')
print('='*55)
print()
print('  Research Gap Addressed:')
print('  → Old systems: predict OR price separately')
print('  → This system: predict AND price together!')
print()
print('  Pricing Logic:')
print('  → Low Congestion    → ₹50  (base)')
print('  → Medium Congestion → ₹80  (base)')
print('  → High Congestion   → ₹120 (base)')
print('  → High Volume surge → ×1.5 multiplier')
print('  → Very slow speed   → +0.3 extra')
print()
print('  Charts saved:')
print('  → static/charts/pricing_scenarios.png')
print('  → static/charts/pricing_rules.png')
print()
print('  Next Step → python app.py → http://127.0.0.1:5000')
print('='*55)